In [ ]:
#!/usr/bin/env python3
"""
YSI Chlorophyll Data Analysis
Analyzes the relationship between Chlorophyll RFU (Relative Fluorescence Units) 
and Chlorophyll concentration (ug/L) from YSI sensor data.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

def load_and_clean_ysi_data(filepath):
    """
    Load YSI data and filter out rows with negative chlorophyll values
    
    Parameters:
    filepath (str): Path to the YSI Excel file
    
    Returns:
    pd.DataFrame: Cleaned dataframe
    """
    # Load the data
    print(f"Loading data from {filepath}...")
    df = pd.read_excel(filepath)
    
    # Display initial data info
    print(f"Initial data shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    
    # Check for chlorophyll columns
    if 'Chl RFU' not in df.columns or 'Chl ug/L' not in df.columns:
        raise ValueError("Required columns 'Chl RFU' and 'Chl ug/L' not found in data")
    
    # Count initial chlorophyll data
    initial_chl_count = df[['Chl RFU', 'Chl ug/L']].notna().all(axis=1).sum()
    print(f"\nRows with both chlorophyll measurements: {initial_chl_count}")
    
    # Filter out rows where either chlorophyll value is negative
    df_filtered = df[(df['Chl RFU'] >= 0) & (df['Chl ug/L'] >= 0)].copy()
    
    # Also remove NaN values in chlorophyll columns
    df_filtered = df_filtered.dropna(subset=['Chl RFU', 'Chl ug/L'])
    
    # Report filtering results
    rows_removed = len(df) - len(df_filtered)
    print(f"Rows removed (negative or NaN values): {rows_removed}")
    print(f"Final data shape: {df_filtered.shape}")
    
    return df_filtered

def plot_chlorophyll_scatter(df, save_path=None):
    """
    Create scatter plot of Chl ug/L vs Chl RFU with regression line
    
    Parameters:
    df (pd.DataFrame): Dataframe with chlorophyll data
    save_path (str): Optional path to save the plot
    """
    # Create figure
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Extract data
    x = df['Chl RFU']
    y = df['Chl ug/L']
    
    # Create scatter plot
    scatter = ax.scatter(x, y, alpha=0.6, s=50, edgecolors='black', linewidth=0.5)
    
    # Calculate regression
    slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
    r_squared = r_value**2
    
    # Plot regression line
    x_line = np.array([x.min(), x.max()])
    y_line = slope * x_line + intercept
    ax.plot(x_line, y_line, 'r-', linewidth=2, label=f'Linear fit: y = {slope:.3f}x + {intercept:.3f}')
    
    # Add R-squared to plot
    ax.text(0.05, 0.95, f'R² = {r_squared:.3f}\nn = {len(x)}', 
            transform=ax.transAxes, fontsize=12,
            verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    # Labels and title
    ax.set_xlabel('Chlorophyll RFU (Relative Fluorescence Units)', fontsize=12)
    ax.set_ylabel('Chlorophyll Concentration (μg/L)', fontsize=12)
    ax.set_title('YSI Sensor Calibration: Chlorophyll RFU vs Concentration', fontsize=14, fontweight='bold')
    ax.legend(fontsize=10)
    
    # Add grid
    ax.grid(True, alpha=0.3)
    
    # Tight layout
    plt.tight_layout()
    
    # Save if path provided
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"\nPlot saved to: {save_path}")
    
    return fig, ax

def generate_statistics(df):
    """
    Generate and print summary statistics for chlorophyll data
    
    Parameters:
    df (pd.DataFrame): Dataframe with chlorophyll data
    """
    print("\n=== Chlorophyll Data Statistics ===")
    
    # Basic statistics
    print(f"\nChlorophyll RFU:")
    print(f"  Range: {df['Chl RFU'].min():.3f} - {df['Chl RFU'].max():.3f}")
    print(f"  Mean: {df['Chl RFU'].mean():.3f} ± {df['Chl RFU'].std():.3f}")
    print(f"  Median: {df['Chl RFU'].median():.3f}")
    
    print(f"\nChlorophyll ug/L:")
    print(f"  Range: {df['Chl ug/L'].min():.3f} - {df['Chl ug/L'].max():.3f}")
    print(f"  Mean: {df['Chl ug/L'].mean():.3f} ± {df['Chl ug/L'].std():.3f}")
    print(f"  Median: {df['Chl ug/L'].median():.3f}")
    
    # Correlation
    correlation = df['Chl RFU'].corr(df['Chl ug/L'])
    print(f"\nPearson correlation coefficient: {correlation:.3f}")
    
    # Date range if available
    if 'Date' in df.columns:
        df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
        valid_dates = df['Date'].dropna()
        if len(valid_dates) > 0:
            print(f"\nDate range: {valid_dates.min().strftime('%Y-%m-%d')} to {valid_dates.max().strftime('%Y-%m-%d')}")
    
    # Site information if available
    if 'Site ID (new)' in df.columns:
        sites = df['Site ID (new)'].value_counts()
        print(f"\nMeasurements by site:")
        for site, count in sites.items():
            print(f"  {site}: {count}")

def plot_residuals(df):
    """
    Create residual plot to check regression assumptions
    
    Parameters:
    df (pd.DataFrame): Dataframe with chlorophyll data
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # Extract data
    x = df['Chl RFU']
    y = df['Chl ug/L']
    
    # Calculate regression
    slope, intercept, _, _, _ = stats.linregress(x, y)
    y_pred = slope * x + intercept
    residuals = y - y_pred
    
    # Residual plot
    ax1.scatter(y_pred, residuals, alpha=0.6)
    ax1.axhline(y=0, color='r', linestyle='--')
    ax1.set_xlabel('Fitted Values (μg/L)', fontsize=11)
    ax1.set_ylabel('Residuals', fontsize=11)
    ax1.set_title('Residual Plot', fontsize=12)
    ax1.grid(True, alpha=0.3)
    
    # Q-Q plot
    stats.probplot(residuals, dist="norm", plot=ax2)
    ax2.set_title('Q-Q Plot', fontsize=12)
    ax2.grid(True, alpha=0.3)
    
    plt.suptitle('Regression Diagnostics', fontsize=14, fontweight='bold')
    plt.tight_layout()
    
    return fig

# Main execution
if __name__ == "__main__":
    # File path
    ysi_file = "CityofSalem_YSI_RawData.xlsx"
    
    try:
        # Load and clean data
        df_ysi = load_and_clean_ysi_data(ysi_file)
        
        # Generate statistics
        generate_statistics(df_ysi)
        
        # Create main scatter plot
        fig1, ax = plot_chlorophyll_scatter(df_ysi, save_path='chlorophyll_rfu_vs_ugl.png')
        
        # Create residual plots
        if len(df_ysi) > 10:  # Only if we have enough data points
            fig2 = plot_residuals(df_ysi)
            plt.savefig('chlorophyll_regression_diagnostics.png', dpi=300, bbox_inches='tight')
        
        # Show plots
        plt.show()
        
        # Optional: Save cleaned data
        output_file = 'ysi_chlorophyll_filtered.csv'
        df_ysi[['Date', 'Site ID (new)', 'Chl RFU', 'Chl ug/L']].to_csv(output_file, index=False)
        print(f"\nFiltered data saved to: {output_file}")
        
        print("\nAnalysis complete!")
        
    except Exception as e:
        print(f"Error: {e}")
        print("Please ensure the file path is correct and the required columns exist.")